In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

In [2]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

df = pd.read_csv(url)
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [3]:
df = df.fillna(0)
df.isna().sum()


model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [4]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
y_full_train = df_full_train["fuel_efficiency_mpg"]
y_train = df_train["fuel_efficiency_mpg"]
y_val = df_val["fuel_efficiency_mpg"]
y_test = df_test["fuel_efficiency_mpg"]

del df_full_train["fuel_efficiency_mpg"]
del df_train["fuel_efficiency_mpg"]
del df_val["fuel_efficiency_mpg"]
del df_test["fuel_efficiency_mpg"]

df_train.columns

Index(['model_year', 'origin', 'fuel_type', 'drivetrain', 'num_doors',
       'engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration'],
      dtype='str')

In [32]:
df_train_dict = df_train.to_dict(orient="records")
df_val_dict = df_val.to_dict(orient="records")

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(df_train_dict)
X_val = dv.transform(df_val_dict)

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=1)
dt_model = dt.fit(X_train, y_train)


In [23]:
from sklearn.tree import export_text

print(export_text(dt_model, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [34]:
from sklearn.metrics import root_mean_squared_error

y_pred = dt_model.predict(X_val)
root_mean_squared_error(y_val, y_pred)
# y_val.shape

2.672633736937051

In [36]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf_model = rf.fit(X_train, y_train)

In [37]:
y_pred = rf_model.predict(X_val)
root_mean_squared_error(y_val, y_pred)

1.8336350509302553

In [39]:
n_estimator = [10, 50, 100, 150]
max_depth =  [10, 15, 20, 25]

for d in max_depth:
    rmse = []
    for n in n_estimator:
        rf = RandomForestRegressor(max_depth=d, n_estimators=n, random_state=1, n_jobs=-1)
        rf_model = rf.fit(X_train, y_train)
        y_pred = rf_model.predict(X_val)
        rmse.append(root_mean_squared_error(y_val, y_pred))
    print(d, np.round(np.mean(rmse), 3))

10 1.757
15 1.785
20 1.783
25 1.787


### Feature Importance Using Trees

In [63]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_model = rf.fit(X_train, y_train)

In [65]:
rf_model.feature_importances_

array([0.05546622, 0.00787275, 0.0200048 , 0.00563607, 0.06100789,
       0.02112758, 0.16071286, 0.01168541, 0.07825357, 0.29582159,
       0.00693892, 0.01972375, 0.00647862, 0.00338878, 0.04223247,
       0.20364873])

In [67]:
importance = pd.Series(
    rf_model.feature_importances_,
    index=dv.get_feature_names_out(),
    name="importance",
).sort_values(ascending=False)

# importance.to_frame().style.format("{:.8f}")
original_features = importance.index.str.split("=", n=1).str[0]
grouped = importance.groupby(original_features).sum().sort_values(ascending=False)

grouped.to_frame().style.format("{:.2%}")

,importance
model_year,29.58%
vehicle_weight,20.36%
fuel_type,19.35%
horsepower,7.83%
engine_displacement,6.10%
acceleration,5.55%
origin,5.21%
drivetrain,3.35%
num_doors,1.97%
num_cylinders,0.69%


In [78]:
import xgboost as xgb

features = list(dv.get_feature_names_out())

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1, 
    
}
watchlist = [(dtrain, 'train'), (dval, 'val')]
model = xgb.train(params=params, dtrain=dtrain, num_boost_round=100, evals=watchlist, verbose_eval=5)

[0]	train-rmse:2.40653	val-rmse:2.46782
[5]	train-rmse:1.59328	val-rmse:1.76216
[10]	train-rmse:1.47125	val-rmse:1.73205
[15]	train-rmse:1.41908	val-rmse:1.73512
[20]	train-rmse:1.39368	val-rmse:1.73766
[25]	train-rmse:1.35837	val-rmse:1.74359


[30]	train-rmse:1.33243	val-rmse:1.74548
[35]	train-rmse:1.29290	val-rmse:1.75198
[40]	train-rmse:1.25326	val-rmse:1.76296
[45]	train-rmse:1.21527	val-rmse:1.77244
[50]	train-rmse:1.18645	val-rmse:1.77517
[55]	train-rmse:1.14797	val-rmse:1.78668
[60]	train-rmse:1.10265	val-rmse:1.79401
[65]	train-rmse:1.06590	val-rmse:1.80143
[70]	train-rmse:1.04604	val-rmse:1.80444
[75]	train-rmse:1.02727	val-rmse:1.80682
[80]	train-rmse:0.99487	val-rmse:1.81308
[85]	train-rmse:0.96970	val-rmse:1.81600
[90]	train-rmse:0.94986	val-rmse:1.82135
[95]	train-rmse:0.91805	val-rmse:1.82544
[99]	train-rmse:0.90106	val-rmse:1.82646


In [ ]:
params['eta'] = 0.1
model = xgb.train(params=params, dtrain=dtrain, num_boost_round=100, evals=watchlist, verbose_eval=5)

[0]	train-rmse:2.74739	val-rmse:2.78635
[5]	train-rmse:2.12109	val-rmse:2.20428
[10]	train-rmse:1.80592	val-rmse:1.93152
[15]	train-rmse:1.64286	val-rmse:1.80346
[20]	train-rmse:1.55847	val-rmse:1.75076
[25]	train-rmse:1.50918	val-rmse:1.72751
[30]	train-rmse:1.47544	val-rmse:1.71667
[35]	train-rmse:1.44978	val-rmse:1.71287
[40]	train-rmse:1.43074	val-rmse:1.71289
[45]	train-rmse:1.41543	val-rmse:1.71310
[50]	train-rmse:1.40368	val-rmse:1.71358
[55]	train-rmse:1.39009	val-rmse:1.71570
[60]	train-rmse:1.37884	val-rmse:1.71688
[65]	train-rmse:1.37172	val-rmse:1.71812
[70]	train-rmse:1.36283	val-rmse:1.71919
[75]	train-rmse:1.35099	val-rmse:1.72057
[80]	train-rmse:1.34578	val-rmse:1.72100
[85]	train-rmse:1.33480	val-rmse:1.72192
[90]	train-rmse:1.32546	val-rmse:1.72303
[95]	train-rmse:1.32292	val-rmse:1.72324
[99]	train-rmse:1.31192	val-rmse:1.72481
